# Targeting the High-Value Risk Zone
### Data Analytics Portfolio — MAN6777 Assessment 3

**Author:** Yasaman Khademi Gilchalan  
**Dataset:** Global AI Impact on Jobs (2010–2025), 5,000 job listings

---

**Managerial question:** Which employee groups should a Chief People Officer (CPO) prioritise for AI-reskilling investment?

We identify the **High-Value Risk Zone** — roles that combine high salary with high automation risk — and test whether AI engagement protects roles from displacement.

**Workflow:** Load → Clean → Engineer features → Quadrant analysis → Hypothesis test → Regression → Visualise

## 1. Setup & Load Data
Import the analytical libraries and load the raw dataset.

In [ ]:
# Core analytics libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

# Display settings
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 5)

# Load the raw dataset
df = pd.read_excel('../data/ai_impact_jobs_2010_2025.xlsx')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Data Inspection & Cleaning
Check data quality before any analysis. We inspect missing values and confirm the core analytical columns are complete.

In [ ]:
# Check for missing values across all columns
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values in any column")

In [ ]:
# The only nulls appear in AI-skill text fields, which are STRUCTURALLY missing
# (they are blank when a role has no AI component). This absence is meaningful
# information, not an error, so we retain these rows rather than dropping/imputing.

# Confirm the core analytical columns used in this study have zero missing values
core_cols = ['salary_usd', 'automation_risk_score', 'ai_intensity_score',
             'ai_mentioned', 'industry', 'seniority_level', 'posting_year']
print("Missing values in core analytical columns:")
print(df[core_cols].isnull().sum())

## 3. Feature Engineering
The raw data contains continuous scores but no decision-ready categories. We engineer a **quadrant classification** — the heart of the analysis — by splitting each role at the dataset medians for salary and automation risk.

In [ ]:
# Define the split points at the dataset medians
salary_median = df['salary_usd'].median()
risk_median   = df['automation_risk_score'].median()
print(f"Median salary: ${salary_median:,.0f}")
print(f"Median automation risk: {risk_median:.2f}")

# Assign each role to one of four strategic quadrants
def assign_quadrant(row):
    high_pay  = row['salary_usd'] >= salary_median
    high_risk = row['automation_risk_score'] >= risk_median
    if high_pay and high_risk:
        return 'High-Value Risk Zone'   # priority for reskilling
    elif high_pay and not high_risk:
        return 'Safe Bet'
    elif not high_pay and high_risk:
        return 'Double Jeopardy'
    else:
        return 'Stable'

df['quadrant'] = df.apply(assign_quadrant, axis=1)

# Readable label for AI engagement (cleaner than True/False in charts)
df['ai_mentioned_label'] = df['ai_mentioned'].map({True: 'AI Mentioned', False: 'No AI'})

print("\nRoles per quadrant:")
print(df['quadrant'].value_counts())

## 4. Quadrant Analysis
We profile each quadrant to understand the scale of the problem and the role of AI engagement.

In [ ]:
# Profile each quadrant: how many roles, and their average characteristics
profile = df.groupby('quadrant').agg(
    roles=('job_id', 'count'),
    avg_salary=('salary_usd', 'mean'),
    avg_risk=('automation_risk_score', 'mean'),
    avg_ai_intensity=('ai_intensity_score', 'mean')
).round(2)
profile['share_%'] = (profile['roles'] / len(df) * 100).round(1)
profile

**Reading the table:** The *High-Value Risk Zone* holds ~22% of roles — well-paid but highly exposed — and shows near-zero AI engagement. The two low-risk quadrants show much higher AI intensity, hinting that AI engagement is protective. We test this next.

## 5. Hypothesis Test — Does AI Engagement Reduce Risk?
**H0:** Roles that mention AI have the same mean automation risk as those that don't.  
**H1:** The means differ.  
We use Welch's t-test (does not assume equal variances).

In [ ]:
# Split into two groups
ai_group    = df[df['ai_mentioned']]['automation_risk_score']
no_ai_group = df[~df['ai_mentioned']]['automation_risk_score']

# Welch's t-test
t_stat, p_value = stats.ttest_ind(ai_group, no_ai_group, equal_var=False)

print(f"Mean risk (AI mentioned): {ai_group.mean():.3f}")
print(f"Mean risk (no AI):        {no_ai_group.mean():.3f}")
print(f"t-statistic: {t_stat:.2f}")
print(f"p-value:     {p_value:.2e}")
print("Result: Reject H0 — the difference is statistically significant" if p_value < 0.05
      else "Result: Fail to reject H0")

In [ ]:
# Pearson correlation between AI intensity and automation risk
r, p_r = stats.pearsonr(df['ai_intensity_score'], df['automation_risk_score'])
print(f"Correlation (AI intensity vs automation risk): r = {r:.3f}, p = {p_r:.2e}")
# Strong negative correlation: more AI engagement -> lower automation risk

## 6. Regression A — What Predicts Automation Risk?
We model automation risk as a function of AI intensity, salary, seniority and industry AI-maturity to identify the strongest protective lever.

In [ ]:
# NOTE: To avoid multicollinearity between the two AI metrics, the boolean
# 'ai_mentioned' is used ONLY in the t-test above. The regressions below use
# only the continuous 'ai_intensity_score' to capture effect size cleanly.

# OLS regression: automation risk as the outcome
model_risk = smf.ols(
    'automation_risk_score ~ ai_intensity_score + salary_usd '
    '+ C(seniority_level) + C(industry_ai_adoption_stage)',
    data=df
).fit()

print(f"R-squared: {model_risk.rsquared:.3f}")
print(f"AI intensity coefficient: {model_risk.params['ai_intensity_score']:.3f} "
      f"(p = {model_risk.pvalues['ai_intensity_score']:.2e})")
# Interpretation: moving a role from 0 to full AI engagement lowers its risk
# score by ~0.71 - the single strongest lever in the model.

## 7. Regression B — The AI Pay Premium
Beyond reducing risk, does AI engagement raise pay? We model salary to quantify the financial upside of reskilling toward AI.

In [ ]:
# OLS regression: salary as the outcome
model_salary = smf.ols(
    'salary_usd ~ ai_intensity_score + C(seniority_level) '
    '+ C(industry_ai_adoption_stage)',
    data=df
).fit()

print(f"R-squared: {model_salary.rsquared:.3f}")
print(f"AI intensity coefficient: ${model_salary.params['ai_intensity_score']:,.0f} "
      f"(p = {model_salary.pvalues['ai_intensity_score']:.2e})")

# Simple AI pay premium
ai_sal   = df[df['ai_mentioned']]['salary_usd'].mean()
noai_sal = df[~df['ai_mentioned']]['salary_usd'].mean()
print(f"\nAvg salary - AI mentioned: ${ai_sal:,.0f}")
print(f"Avg salary - no AI:        ${noai_sal:,.0f}")
print(f"AI pay premium: ${ai_sal - noai_sal:,.0f} (+{(ai_sal/noai_sal - 1)*100:.0f}%)")
# Holding seniority & industry constant, full AI engagement is associated with
# ~$36.5K higher salary. Reskilling moves staff into a higher-paid, safer tier.

## 8. Visualisations
The figures below are also rebuilt in Power BI for the interactive dashboard. Here we reproduce them in matplotlib for the portfolio record.

### Figure 1 — The High-Value Risk Zone Matrix

In [ ]:
colors = {'Safe Bet':'#1D9E75', 'High-Value Risk Zone':'#D85A30',
          'Double Jeopardy':'#E24B4A', 'Stable':'#888780'}

fig, ax = plt.subplots(figsize=(9, 5.5))
for q, c in colors.items():
    sub = df[df['quadrant'] == q]
    ax.scatter(sub['automation_risk_score'], sub['salary_usd'],
               c=c, s=12, alpha=0.35, edgecolors='none', label=q)
ax.axvline(risk_median, color='#444', ls='--', lw=1)
ax.axhline(salary_median, color='#444', ls='--', lw=1)
ax.set_xlabel('Automation Risk Score')
ax.set_ylabel('Salary (USD)')
ax.set_title('Figure 1 — Mapping 5,000 Jobs by Pay vs. Automation Risk')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Figure 2 — Workforce Distribution Across Quadrants

In [ ]:
counts = df['quadrant'].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=[colors[q] for q in counts.index],
       wedgeprops={'width': 0.45})  # donut
ax.set_title('Figure 2 — Share of Roles by Strategic Quadrant')
plt.tight_layout(); plt.show()

### Figure 3 — AI Skills as a Shield

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['ai_intensity_score'], df['automation_risk_score'],
           c='#534AB7', s=10, alpha=0.25, edgecolors='none')
z = np.polyfit(df['ai_intensity_score'], df['automation_risk_score'], 1)
xx = np.linspace(0, 0.95, 100)
ax.plot(xx, z[0]*xx + z[1], color='#D85A30', lw=2.5, label=f'Trend (r = {r:.3f})')
ax.set_xlabel('AI Intensity Score')
ax.set_ylabel('Automation Risk Score')
ax.set_title('Figure 3 — Higher AI Intensity Predicts Lower Automation Risk')
ax.legend(); plt.tight_layout(); plt.show()

### Figure 4 — The Great Shift (2010–2025)

In [ ]:
yearly = df.groupby('posting_year').agg(
    avg_ai=('ai_intensity_score', 'mean'),
    avg_risk=('automation_risk_score', 'mean')
)
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax1.plot(yearly.index, yearly['avg_ai'], color='#534AB7', lw=2.5, marker='o', label='AI Intensity')
ax2.plot(yearly.index, yearly['avg_risk'], color='#D85A30', lw=2.5, marker='s', label='Automation Risk')
ax1.set_xlabel('Year'); ax1.set_ylabel('Avg AI Intensity', color='#534AB7')
ax2.set_ylabel('Avg Automation Risk', color='#D85A30')
ax1.set_title('Figure 4 — AI Intensity Rises as Automation Risk Falls')
plt.tight_layout(); plt.show()

### Figure 5 — Exposure by Industry

In [ ]:
trap = df[df['quadrant'] == 'High-Value Risk Zone']
by_industry = (trap['industry'].value_counts() / df['industry'].value_counts() * 100
               ).round(1).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(by_industry.index, by_industry.values, color='#D85A30')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_xlabel('% of roles in the High-Value Risk Zone')
ax.set_title('Figure 5 — Which Industries Are Most Exposed?')
plt.tight_layout(); plt.show()

### Figure 6 — Risk by Industry AI-Maturity

In [ ]:
maturity = df.groupby('industry_ai_adoption_stage')['automation_risk_score'].mean().reindex(
    ['Emerging', 'Growing', 'Mature'])
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(maturity.index, maturity.values, color=['#E24B4A', '#E8A33D', '#1D9E75'])
ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set_ylabel('Avg Automation Risk')
ax.set_title('Figure 6 — Mature-AI Industries Carry Lower Risk')
plt.tight_layout(); plt.show()

## 9. Conclusion
- **22.4%** of roles sit in the High-Value Risk Zone (high pay, high automation risk).
- AI engagement is strongly protective: AI-mentioned roles average **0.25** risk vs **0.75** (p < 0.001); correlation **r = −0.875**.
- Regression A: AI intensity is the dominant predictor of lower risk (R² = 0.77).
- Regression B: full AI engagement is associated with **+$36.5K** salary; AI-mentioned roles earn a **58%** premium.

**Recommendation:** Prioritise reskilling the High-Value Risk Zone toward AI skills — it both lowers displacement risk and lifts roles into a higher-paid, more durable tier. Start with the most-exposed industries (Government, Healthcare, Agriculture).

*This notebook is fully reproducible: clone the repo, place the dataset in `../data/`, and Run All.*

## 8. Correlation Matrix
A correlation matrix quantifies how the four key numeric variables move together. This addresses two questions at once: does AI engagement lower risk, and does it also relate to pay and pay growth?

In [ ]:
# Select the four numeric variables
cols = ['ai_intensity_score', 'automation_risk_score',
        'salary_usd', 'salary_change_vs_prev_year_percent']
labels = ['AI Intensity', 'Automation Risk', 'Salary', 'Salary Growth %']

# Compute Pearson correlation matrix
corr = df[cols].corr()
print(corr.round(3))

# Draw heatmap
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_yticklabels(labels)
for i in range(len(cols)):
    for j in range(len(cols)):
        v = corr.iloc[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                color='white' if abs(v) > 0.5 else 'black', fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Correlation (r)')
ax.set_title('Correlation Matrix: AI, Risk, Salary & Salary Growth', fontweight='bold')
plt.tight_layout(); plt.show()

# Key takeaways:
# AI intensity vs automation risk: r = -0.88 (strong negative - AI protects)
# AI intensity vs salary growth:  r = +0.67 (AI roles grow pay faster)
# automation risk vs salary growth: r = -0.68 (risky roles stagnate)